# Fine-Tuning Lightweight MobileViT on Combined Kaggle Tomato Disease Datasets

This Google Colab notebook downloads two tomato leaf disease datasets from Kaggle using `kagglehub`, combines and aligns the shared disease classes, and fine-tunes a lightweight Vision Transformer (`apple/mobilevit-xx-small`, ~1.3M parameters) using Hugging Face `transformers` and PyTorch.

### Step 1: Install Dependencies & Check GPU

In [ ]:
!nvidia-smi
!pip install -q kagglehub transformers datasets evaluate accelerate torchvision scikit-learn matplotlib pillow

### Step 2: Download Both Datasets via kagglehub

In [ ]:
import kagglehub

# Download PlantVillage (full plant dataset containing tomato classes)
pv_path = kagglehub.dataset_download("adilmubashirchaudhry/plant-village-dataset")
print("PlantVillage Dataset Path:", pv_path)

# Download 6-Class Tomato Leaf Disease Dataset
sc_path = kagglehub.dataset_download("syedhashirali260/tomato-leaf-disease-dataset-6-classes")
print("6-Class Tomato Dataset Path:", sc_path)

### Step 3: Discover, Align Similar Classes, and Merge Datasets Class-Wise

In [ ]:
import os
import re
import random
import shutil
from pathlib import Path
from typing import Dict, List

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def normalize_class_name(folder_name: str) -> str:
    """Normalizes folder names into unified canonical keys."""
    s = re.sub(r"^tomato\s*[-_]*\s*", "", folder_name, flags=re.IGNORECASE)
    s = re.sub(r"[\s\-_]+", "_", s).strip("_").lower()
    return s

def find_tomato_images(root_dir: str, require_tomato_filter: bool = False) -> Dict[str, List[str]]:
    class_to_images = {}
    for p in Path(root_dir).rglob("*"):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS:
            parent = p.parent.name
            if require_tomato_filter and "tomato" not in str(p).lower():
                continue
            cname = normalize_class_name(parent)
            class_to_images.setdefault(cname, []).append(str(p))
    return class_to_images

# Scan both datasets
ds1_imgs = find_tomato_images(pv_path, require_tomato_filter=True)
ds2_imgs = find_tomato_images(sc_path, require_tomato_filter=False)

# Find intersecting classes
common_classes = sorted(list(set(ds1_imgs.keys()).intersection(set(ds2_imgs.keys()))))
print(f"\nSuccessfully identified {len(common_classes)} overlapping classes:")
for c in common_classes:
    print(f"  • {c:20} -> DS1: {len(ds1_imgs[c])} | DS2: {len(ds2_imgs[c])} | Total: {len(ds1_imgs[c]) + len(ds2_imgs[c])}")

# Create merged directory with 80% train / 20% validation split
merged_dir = "./combined_tomato_dataset"
train_dir = os.path.join(merged_dir, "train")
val_dir = os.path.join(merged_dir, "validation")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(val_dir, exist_ok=True)

random.seed(42)
for c in common_classes:
    all_imgs = ds1_imgs[c] + ds2_imgs[c]
    random.shuffle(all_imgs)
    split_idx = int(len(all_imgs) * 0.8)
    
    c_train = os.path.join(train_dir, c)
    c_val = os.path.join(val_dir, c)
    os.makedirs(c_train, exist_ok=True)
    os.makedirs(c_val, exist_ok=True)
    
    for idx, src in enumerate(all_imgs[:split_idx]):
        dst = os.path.join(c_train, f"{c}_{idx:05d}{Path(src).suffix}")
        if not os.path.exists(dst):
            try: os.symlink(os.path.abspath(src), dst)
            except OSError: shutil.copy2(src, dst)
            
    for idx, src in enumerate(all_imgs[split_idx:]):
        dst = os.path.join(c_val, f"{c}_{idx:05d}{Path(src).suffix}")
        if not os.path.exists(dst):
            try: os.symlink(os.path.abspath(src), dst)
            except OSError: shutil.copy2(src, dst)

print(f"\nDataset merged into '{merged_dir}' ready for training!")

### Step 4: Preview Sample Images from Each Disease Class

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

fig, axes = plt.subplots(1, len(common_classes), figsize=(18, 4))
for idx, c in enumerate(common_classes):
    cls_dir = os.path.join(train_dir, c)
    sample_file = os.listdir(cls_dir)[0]
    img = Image.open(os.path.join(cls_dir, sample_file))
    axes[idx].imshow(img)
    axes[idx].set_title(c.replace('_', ' ').title(), fontsize=10, weight='bold')
    axes[idx].axis("off")
plt.tight_layout()
plt.show()

### Step 5: Data Loading & Vision Transformer Preprocessing

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoImageProcessor
from torchvision.transforms import (
    Compose, Resize, CenterCrop, RandomResizedCrop, RandomHorizontalFlip,
    RandomRotation, ColorJitter, ToTensor, Normalize
)

# Load image dataset
dataset = load_dataset("imagefolder", data_dir=merged_dir)

labels = dataset["train"].features["label"].names
num_labels = len(labels)
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}

# Model selection (MobileViT XXS: ~1.3M parameters)
MODEL_CKPT = "apple/mobilevit-xx-small"
image_processor = AutoImageProcessor.from_pretrained(MODEL_CKPT)

size = (image_processor.size["height"], image_processor.size["width"]) if "height" in image_processor.size else (224, 224)
mean = getattr(image_processor, "image_mean", [0.485, 0.456, 0.406])
std = getattr(image_processor, "image_std", [0.229, 0.224, 0.225])

# Data augmentation transforms
train_tf = Compose([
    RandomResizedCrop(size, scale=(0.8, 1.0)),
    RandomHorizontalFlip(),
    RandomRotation(15),
    ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    ToTensor(),
    Normalize(mean=mean, std=std)
])

val_tf = Compose([
    Resize(size),
    CenterCrop(size),
    ToTensor(),
    Normalize(mean=mean, std=std)
])

def preprocess_train(batch):
    batch["pixel_values"] = [train_tf(img.convert("RGB")) for img in batch["image"]]
    return batch

def preprocess_val(batch):
    batch["pixel_values"] = [val_tf(img.convert("RGB")) for img in batch["image"]]
    return batch

dataset["train"].set_transform(preprocess_train)
dataset["validation"].set_transform(preprocess_val)

def collate_fn(batch):
    return {
        "pixel_values": torch.stack([x["pixel_values"] for x in batch]),
        "labels": torch.tensor([x["label"] for x in batch], dtype=torch.long)
    }

### Step 6: Load Model & Training Setup

In [ ]:
import evaluate
import numpy as np
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer

model = AutoModelForImageClassification.from_pretrained(
    MODEL_CKPT,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)

metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, targets = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = metric_acc.compute(predictions=preds, references=targets)["accuracy"]
    macro_f1 = metric_f1.compute(predictions=preds, references=targets, average="macro")["f1"]
    return {"accuracy": acc, "f1_macro": macro_f1}

training_args = TrainingArguments(
    output_dir="./tomato_mobilevit_results",
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    learning_rate=3e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    num_train_epochs=8,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    logging_steps=25,
    fp16=torch.cuda.is_available(),
    remove_unused_columns=False,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=image_processor,
    data_collator=collate_fn,
    compute_metrics=compute_metrics
)

### Step 7: Train the Model

In [ ]:
# Run training
trainer.train()

# Evaluate on validation split
eval_results = trainer.evaluate()
print("\nValidation Results:", eval_results)

# Save final artifacts
trainer.save_model("./tomato_mobilevit_final")
image_processor.save_pretrained("./tomato_mobilevit_final")
print("Model saved to ./tomato_mobilevit_final")

### Step 8: Test Inference on a Leaf Image

In [ ]:
from transformers import pipeline
import glob

# Pick a test image from validation set
test_images = glob.glob("./combined_tomato_dataset/validation/*/*")
sample_image_path = test_images[0]

classifier = pipeline("image-classification", model="./tomato_mobilevit_final")
predictions = classifier(sample_image_path)

print(f"Test Image: {sample_image_path}")
print(f"Predicted Diagnosis: {predictions[0]['label']} ({predictions[0]['score']*100:.2f}% confidence)")

# Display image with prediction
img = Image.open(sample_image_path)
plt.figure(figsize=(4, 4))
plt.imshow(img)
plt.title(f"{predictions[0]['label']} ({predictions[0]['score']*100:.1f}%)")
plt.axis("off")
plt.show()